# 10. 딥러닝 다중분류 - 실무전용 문제 모음

> **실무 전용 노트북** - 이론 설명 없이 TODO 문제만 모았습니다. (관련 이론 노트북: 10_딥러닝_분류_다중.ipynb)
이미 개념은 이해했다는 전제로, 손으로 더 많이 풀어보며 완전히 몸에 익히는 것이 목표입니다. 정답을 먼저 보지 말고 반드시 스스로 코드를 작성해본 뒤 펼쳐서 비교하세요.

이론 노트북에서는 wine 데이터로 연습했다면, 여기서는 **동일 wine 데이터를 다른 구조의 모델**로 다시 연습해봅니다.

## 목차
1. 모델 설계 연습 (더 깊은 구조)
2. 모델 학습 & 예측 후처리 연습
3. 성능 평가 연습

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf

tf.random.set_seed(100)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

wine = pd.read_csv('../data/wine.csv')
X = wine.drop(columns=['target'])
y = wine['target']
class_num = y.nunique()
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)
X_train.shape

## 1. 모델 설계 연습 (더 깊은 구조)

**문제 1.** 입력층 32(relu)→16(relu)→8(relu)→출력 class_num(softmax)인 `model`을 만들어 `sparse_categorical_crossentropy`로 컴파일하세요.

In [ ]:
# 여기에 코드를 작성하세요


<details>
<summary>✅ 정답 보기</summary>

```python
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
model = Sequential()
model.add(Dense(32, activation='relu', input_shape=(X_train_s.shape[1],)))
model.add(Dense(16, activation='relu'))
model.add(Dense(8, activation='relu'))  # 은닉층을 여러 겹 쌓을수록 더 복잡한 패턴을 학습할 수 있지만 과적합 위험도 커짐
model.add(Dense(class_num, activation='softmax'))  # 다중분류 출력층: 노드 수=클래스 수, softmax
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.summary()
```

</details>

## 2. 모델 학습 & 예측 후처리 연습

**문제 2.** `EarlyStopping(patience=10)`로 학습시킨 뒤 `np.argmax`로 예측 클래스를 구하세요.

In [ ]:
# 여기에 코드를 작성하세요


<details>
<summary>✅ 정답 보기</summary>

```python
from tensorflow.keras.callbacks import EarlyStopping
es = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
model.fit(X_train_s, y_train, epochs=100, batch_size=16, validation_data=(X_test_s, y_test), callbacks=[es], verbose=0)
pred_label = np.argmax(model.predict(X_test_s), axis=1)  # predict가 반환하는 클래스별 확률 벡터에서 argmax로 최종 클래스를 뽑아냄
print(pred_label[:10])
```

</details>

## 3. 성능 평가 연습

**문제 3.** accuracy와 confusion matrix를 확인하세요.

In [ ]:
# 여기에 코드를 작성하세요


<details>
<summary>✅ 정답 보기</summary>

```python
from sklearn.metrics import accuracy_score, confusion_matrix
print(accuracy_score(y_test, pred_label))
cm = confusion_matrix(y_test, pred_label)  # 클래스 개수만큼 NxN 행렬이 반환됨
sns.heatmap(cm, annot=True, fmt='d', cmap='Purples')
plt.show()
```

</details>